## STEP 0 
Import Libraries, Connect to Snowflake, & Initialize NLP Pipelines

In [3]:
from datasets import Dataset
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import torch
from transformers import pipeline

# this basically means "smoke em if you got em" where the "em" is NVIDIA GPU
DEVICE = 0 if torch.cuda.is_available() else -1

SF_USR = os.getenv('SF_USR')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')

# connect to database and init a cursor for querying
xct_params = {
    "user":                 os.getenv('SF_USR')
   ,"account":              os.getenv('SF_ID')
   ,"warehouse":            os.getenv('SF_WH')
   ,"database":             os.getenv('SF_DB')
   ,"schema":               os.getenv('SF_SC')
   ,"role":                 os.getenv('SF_RL')
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}
SF_XCT = snowflake.connector.connect(**xct_params)
CSR = SF_XCT.cursor()

# sentiment analyzer doo-dad instantiation
PIPL_SNT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=DEVICE,
    truncation=True,
    max_length = 512 
)
## This sentiment pipeline returns labels like ['LABEL_0', 'LABEL_1', 'LABEL_2']
## instead of ['Negative', 'Neutral', 'Positive']
## The below-linked mapping indicates which model-labels match to which
## human-understandable terms. 
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=256 
)

Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


## STEP 1
Ingest Post-Text into Memory

In [ ]:
query = f"""
select content_id
      ,usa_timestamp as post_created_usa_timestamp
      ,post_text
from {SF_DB}.{SF_SC}.firehose_processed
where (first_detected_language = 'English'
   or  first_detected_language is null
   )
  and post_created_usa_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                     and to_timestamp_tz('2025-05-31 23:59:59+0000')
;
"""
# and post_created_usa_timestamp >= (select nvl(max(post_created_usa_timestamp), '1900-00-00 00:00:00+0000)
#                                    from {SF_DB}.{SF_SC}.firehose_nlp_labeled) 
#
CSR.execute(query)
total_rows = len(CSR.execute(query).fetch_pandas_all()) # execute once just to get total rows
print(f"{(total_rows):,} downloaded. Processing in batches...\n\n")

# again to start iterating over batches
CSR.execute(query)

c=0
current_pcnt = 0
for batch in CSR.fetch_pandas_batches():
    c+=1
    current_pcnt+=round((len(batch)/total_rows)*100, 1)
    print(f"\n{len(batch):,} rows downloaded from batch {(c):,} ({(current_pcnt):,.1f}% of total rows)")

    batch_dataset = Dataset.from_pandas(batch[['POST_TEXT']], preserve_index=False)

    # execute NER analysis 
    ner_output = PIPL_NER(batch_dataset['POST_TEXT'])

    # Add this col now to match schema-- will populate in the next step
    batch['SENTIMENT_ANALYSIS'] = None
    batch['NER_ANALYSIS']       = ner_output

    # match schema ordinal
    batch = batch[['CONTENT_ID','POST_CREATED_USA_TIMESTAMP','SENTIMENT_ANALYSIS','NER_ANALYSIS','POST_TEXT']]
    write_pandas(SF_XCT, batch
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
            ,use_logical_type = True
            ,auto_create_table = False
            ,overwrite = False
           )
    print(f"{len(batch):,} rows from batch written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")

562,204 downloaded. Processing in batches...



572 rows downloaded from batch 1  (0.1% of total rows)
572 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP

1,677 rows downloaded from batch 2  (0.4% of total rows)
1,677 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP

2,941 rows downloaded from batch 3  (0.9% of total rows)
2,941 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP

4,576 rows downloaded from batch 4  (1.7% of total rows)
4,576 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP

11,584 rows downloaded from batch 5  (3.8% of total rows)


## STEP 2

Apply Named-Entity Recognition (NER) and write to a stash table (`INT_FIREHOSE_NLP`)

In [ ]:
ner_output = PIPL_NER(DATA['POST_TEXT'].tolist())

# Add this col now to match schema-- will populate in the next step
DATA['SENTIMENT_ANALYSIS'] = None
DATA['NER_ANALYSIS']       = ner_output

write_pandas(SF_XCT, DATA
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")

KeyboardInterrupt: 

## STEP 3
Re-read data and apply Sentiment analysis, then writeback to stash

In [ ]:
query = f"select POST_TEXT,CONTENT_ID from {SF_DB}.{SF_SC}.int_firehose_nlp"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

sentiment_output = PIPL_SNT(DATA['POST_TEXT'].tolist())
DATA['SENTIMENT_ANALYSIS'] = sentiment_output

query=f"""
create temp table if not exists {SF_DB}.{SF_SC}.TMP_MERGE_SRC (
 POST_TEXT VARCHAR
,CONTENT_ID VARCHAR
,SENTIMENT_ANALYSIS VARIANT
)"""
CSR.execute(query)

write_pandas(SF_XCT, DATA
            ,table_name = 'TMP_MERGE_SRC'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.TMP_MERGE_SRC")

query=f"""
merge into {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP tgt
using {SF_DB}.{SF_SC}.TMP_MERGE_SRC src
   on src.content_id = tgt.content_id
when matched then update 
set tgt.SENTIMENT_ANALYSIS = src.SENTIMENT_ANALYSIS
"""
print(f"{(CSR.fetchone()[1]):,} rows updated in INT_FIREHOSE_NLP.SENTIMENT_ANALYSIS, using TMP_MERGE_SRC")

## STEP 4
 Retrieve stashed data, blow it out into many processed rows per-post, then insert.

In [ ]:
query = f"select * from {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

query = f"""
insert into {SF_DB}.{SF_SC}.firehose_nlp_labeled
with src as (
select a.content_id
      ,a.post_created_usa_timestamp
      ,b.readable_label_name as sentiment_detected_label
      ,cast(sentiment_analysis:score as number(5,4)) as sentiment_confidence_score
      ,row_number() over (
       partition by content_id
       order     by post_created_usa_timestamp, trim(a2.value:word, '"')
       ) as post_entity_number
      ,trim(a2.value:entity_group, '"') as ner_detected_group
      ,trim(a2.value:word, '"') as ner_detected_entity
      ,cast(a2.value:score as number(5,4)) as ner_confidence_score
from {SF_DB}.{SF_SC}.int_firehose_nlp a
left join table(flatten(input => parse_json(a.ner_analysis))) a2
left join {SF_DB}.{SF_SC}.label_map_roberta_base_sentiment b
       on trim(a.sentiment_analysis:label, '"') = b.model_label_name
)

select sha2(nvl(to_char(content_id), 'NULL') 
         || '||' 
         || nvl(to_char(post_entity_number), 'NULL')
       ) as analysis_id
      ,*
from src 
order by content_id
        ,post_created_usa_timestamp
        ,ner_detected_entity
"""
CSR.execute(query)

## Closing
Clear the stash, shut off the Snowflake Connection

In [ ]:
query = f'truncate table {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP'
CSR.execute(query)
SF_XCT.close()